# Data Cleaning & EDA

In [ ]:
import sys; from pathlib import Path; sys.path.append(str(Path.cwd().parent / 'python'))

In [ ]:
import pandas as pd
from config import RAW_DIR, PROCESSED_DIR
from utils import logger

def clean_all_datasets() -> None:
    cust = pd.read_csv(RAW_DIR / "customers_raw.csv").drop_duplicates(subset=["customer_id"], keep="first").copy()
    cust["acquisition_channel"] = cust["acquisition_channel"].astype(str).str.strip()
    cust["gender"] = cust["gender"].fillna("Unspecified")
    cust["age"] = cust["age"].clip(lower=18, upper=95)
    cust.to_csv(PROCESSED_DIR / "customers_clean.csv", index=False)

    prod = pd.read_csv(RAW_DIR / "products_raw.csv").drop_duplicates(subset=["product_id"], keep="first").copy()
    prod["unit_price"] = prod[["unit_price", "unit_cost"]].max(axis=1)
    prod.to_csv(PROCESSED_DIR / "products_clean.csv", index=False)

    camp = pd.read_csv(RAW_DIR / "campaigns_raw.csv").drop_duplicates(subset=["campaign_id"], keep="first").copy()
    camp.to_csv(PROCESSED_DIR / "campaigns_clean.csv", index=False)

    orders = pd.read_csv(RAW_DIR / "orders_raw.csv").drop_duplicates(subset=["order_id"], keep="first").copy()
    orders["payment_method"] = orders["payment_method"].fillna("Credit Card")
    orders["order_date"] = pd.to_datetime(orders["order_date"])
    orders = orders[orders["customer_id"].isin(cust["customer_id"])]
    orders.to_csv(PROCESSED_DIR / "orders_clean.csv", index=False)

    items = pd.read_csv(RAW_DIR / "order_items_raw.csv").drop_duplicates(subset=["order_item_id"], keep="first").copy()
    items = items[items["order_id"].isin(orders["order_id"]) & items["product_id"].isin(prod["product_id"])]
    items["quantity"] = items["quantity"].clip(lower=1)
    items.to_csv(PROCESSED_DIR / "order_items_clean.csv", index=False)
    logger.info("All 5 cleaned datasets saved to data/processed/.")

if __name__ == "__main__":
    clean_all_datasets()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from utils import load_clean_tables, save_plot, logger

def run_eda() -> None:
    tables = load_clean_tables()
    orders, items = tables["orders"], tables["order_items"]
    completed = orders[orders["order_status"] == "Completed"].copy()
    completed = completed.merge(items.groupby("order_id", as_index=False)["line_margin"].sum(), on="order_id", how="left")
    completed["year_month"] = completed["order_date"].dt.to_period("M").astype(str)

    monthly = completed.groupby("year_month", as_index=False).agg(revenue=("total_amount", "sum"), margin=("line_margin", "sum"))
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(monthly["year_month"], monthly["revenue"], marker="o", label="Net Revenue ($)")
    ax.plot(monthly["year_month"], monthly["margin"], marker="s", label="Gross Profit ($)")
    ax.set_title("Monthly Net Revenue & Gross Margin Trend", fontweight="bold")
    ax.tick_params(axis="x", rotation=45)
    ax.legend()
    save_plot(fig, "01_monthly_revenue_trend.png")

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.boxplot(data=completed, x="device_type", y="total_amount", showfliers=False, ax=ax)
    ax.set_title("Order Value Distribution by Device Type", fontweight="bold")
    save_plot(fig, "02_aov_by_device.png")
    logger.info("EDA complete.")

if __name__ == "__main__":
    run_eda()
